In [21]:
import os
import os.path
import pandas as pd
import numpy as np

In [22]:
datadir = "data"
data = os.path.join(datadir, "adult.data")
df = pd.read_csv(data)
df.columns = ["age", "workclass", "fnlwgt", "education", "education-num", 
              "marital-status", "occupation", "relationship", "race", "sex", 
              "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"]

In [23]:
df.isna().sum()

age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64

In [24]:
columns = df.columns

for col in columns:
    print("Column:", col, df[col].unique())
    print()

Column: age [50 38 53 28 37 49 52 31 42 30 23 32 40 34 25 43 54 35 59 56 19 39 20 45
 22 48 21 24 57 44 41 29 18 47 46 36 79 27 67 33 76 17 55 61 70 64 71 68
 66 51 58 26 60 90 75 65 77 62 63 80 72 74 69 73 81 78 88 82 83 84 85 86
 87]

Column: workclass [' Self-emp-not-inc' ' Private' ' State-gov' ' Federal-gov' ' Local-gov'
 ' ?' ' Self-emp-inc' ' Without-pay' ' Never-worked']

Column: fnlwgt [ 83311 215646 234721 ...  34066  84661 257302]

Column: education [' Bachelors' ' HS-grad' ' 11th' ' Masters' ' 9th' ' Some-college'
 ' Assoc-acdm' ' Assoc-voc' ' 7th-8th' ' Doctorate' ' Prof-school'
 ' 5th-6th' ' 10th' ' 1st-4th' ' Preschool' ' 12th']

Column: education-num [13  9  7 14  5 10 12 11  4 16 15  3  6  2  1  8]

Column: marital-status [' Married-civ-spouse' ' Divorced' ' Married-spouse-absent'
 ' Never-married' ' Separated' ' Married-AF-spouse' ' Widowed']

Column: occupation [' Exec-managerial' ' Handlers-cleaners' ' Prof-specialty'
 ' Other-service' ' Adm-clerical' ' Sales' ' Cra

In [25]:
# ----------------------------
# AGE
# ----------------------------
def generalize_age(age, level=3):
    if level == 2:  # 5-year bins
        return f"{(age // 5) * 5}-{(age // 5) * 5 + 4}"
    elif level == 3:  # 10-year bins
        return f"{(age // 10) * 10}-{(age // 10) * 10 + 9}"
    elif level == 4:  # broad groups
        if age <= 24: return "Youth"
        elif age <= 44: return "Young Adult"
        elif age <= 64: return "Middle-aged"
        else: return "Senior"
    return age  # raw

# ----------------------------
# WORKCLASS
# ----------------------------
def generalize_workclass(wc, level=3):
    wc = wc.strip()
    if level == 2:
        mapping = {
            'Self-emp-not-inc': 'Self-employed',
            'Self-emp-inc': 'Self-employed',
            'State-gov': 'Government',
            'Local-gov': 'Government',
            'Federal-gov': 'Government',
            'Private': 'Private',
            'Without-pay': 'Other',
            'Never-worked': 'Other',
            '?': 'Other'
        }
        return mapping.get(wc, wc)
    elif level == 3:
        if wc in ['Private']: return 'Private'
        if wc in ['Self-emp-not-inc','Self-emp-inc']: return 'Self-employed'
        if wc in ['State-gov','Local-gov','Federal-gov']: return 'Government'
        return 'Other'
    elif level == 4:
        if wc in ['Private','Self-emp-not-inc','Self-emp-inc','State-gov','Local-gov','Federal-gov']:
            return 'Employed'
        return 'Not in labor force/Unknown'
    return wc

# ----------------------------
# EDUCATION
# ----------------------------
def generalize_education(ed, level=3):
    ed = ed.strip()
    if level == 2:
        primary = ['Preschool','1st-4th','5th-6th','7th-8th']
        secondary = ['9th','10th','11th','12th','HS-grad']
        postsecondary = ['Some-college','Assoc-acdm','Assoc-voc']
        tertiary = ['Bachelors','Masters','Doctorate','Prof-school']
        if ed in primary: return 'Primary'
        if ed in secondary: return 'Secondary'
        if ed in postsecondary: return 'Postsecondary'
        if ed in tertiary: return 'Tertiary'
    elif level == 3:
        if ed in ['Preschool','1st-4th','5th-6th','7th-8th','9th','10th','11th','12th','HS-grad']:
            return 'Low'
        if ed in ['Some-college','Assoc-acdm','Assoc-voc']:
            return 'Medium'
        if ed in ['Bachelors','Masters','Doctorate','Prof-school']:
            return 'High'
    elif level == 4:
        if ed in ['Bachelors','Masters','Doctorate','Prof-school']:
            return 'High'
        return 'Low'
    return ed

# ----------------------------
# HOURS-PER-WEEK
# ----------------------------
def generalize_hours(h, level=3):
    if level == 2:
        if h < 20: return "0–19"
        elif h < 40: return "20–39"
        elif h < 60: return "40–59"
        else: return "60+"
    elif level == 3:
        if h < 35: return "Part-time"
        elif h < 50: return "Full-time"
        else: return "Overtime"
    elif level == 4:
        return "Standard" if h == 40 else "Nonstandard"
    return h

# ----------------------------
# NATIVE-COUNTRY
# ----------------------------
def generalize_country(c, level=3):
    c = c.strip()
    if level == 2:
        NA = ['United-States','Canada','Puerto-Rico','Outlying-US(Guam-USVI-etc)','Honduras','Mexico','Cuba','Jamaica','Trinadad&Tobago']
        EU = ['England','Germany','Italy','Poland','France','Yugoslavia','Scotland','Greece','Ireland','Hungary','Holand-Netherlands']
        AS = ['India','Iran','Philippines','Cambodia','Thailand','Laos','Taiwan','China','Japan','Vietnam','Hong']
        LATAM = ['Columbia','Ecuador','Haiti','Dominican-Republic','El-Salvador','Guatemala','Nicaragua','Peru','South']
        if c in NA: return "North America"
        if c in EU: return "Europe"
        if c in AS: return "Asia"
        if c in LATAM: return "Latin America"
        if c == '?': return "Unknown"
        return "Other"
    elif level == 3:
        if c == "United-States": return "US"
        if c == "?": return "Unknown"
        return "Non-US"
    elif level == 4:
        return "Domestic" if c == "United-States" else "Foreign/Unknown"
    return c


# ----------------------------
# MARITAL-STATUS
# ----------------------------
def generalize_marital(ms, level=3):
    ms = ms.strip()
    if level == 2:
        if "Married" in ms:
            return "Married"
        else:
            return "Not married"
    elif level == 3:
        if "Married" in ms:
            return "With partner"
        else:
            return "Without partner"
    return ms  # raw

# ----------------------------
# OCCUPATION
# ----------------------------
def generalize_occupation(occ, level=3):
    occ = occ.strip()
    if level == 2:
        white_collar = ['Exec-managerial','Prof-specialty','Adm-clerical','Sales','Tech-support']
        blue_collar  = ['Handlers-cleaners','Craft-repair','Transport-moving','Machine-op-inspct','Farming-fishing']
        services     = ['Other-service','Protective-serv','Priv-house-serv']
        other        = ['?','Armed-Forces']

        if occ in white_collar: return "White-collar"
        if occ in blue_collar: return "Blue-collar"
        if occ in services: return "Services"
        if occ in other: return "Other/Unknown"
    elif level == 3:
        if occ in ['Exec-managerial','Prof-specialty','Adm-clerical','Sales','Tech-support']:
            return "White-collar"
        if occ in ['Handlers-cleaners','Craft-repair','Transport-moving','Machine-op-inspct','Farming-fishing']:
            return "Blue-collar"
        return "Other"
    elif level == 4:
        if occ in ['Exec-managerial','Prof-specialty','Adm-clerical','Sales','Tech-support',
                   'Handlers-cleaners','Craft-repair','Transport-moving','Machine-op-inspct','Farming-fishing',
                   'Other-service','Protective-serv','Priv-house-serv']:
            return "Employed"
        return "Other/Unknown"
    return occ

# ----------------------------
# RELATIONSHIP
# ----------------------------
def generalize_relationship(r, level=3):
    r = r.strip()
    if level == 2:
        if r in ['Husband','Wife','Own-child','Other-relative']:
            return "In-family"
        else:
            return "Not-in-family"
    elif level == 3:
        if r in ['Husband','Wife','Own-child','Other-relative']:
            return "With family"
        else:
            return "Without family"
    return r

# ----------------------------
# RACE
# ----------------------------
def generalize_race(r, level=3):
    r = r.strip()
    if level == 2:
        if r == "White": return "White"
        if r == "Black": return "Black"
        if r == "Asian-Pac-Islander": return "Asian"
        return "Other"
    elif level == 3:
        if r == "White": return "Majority"
        return "Minority"
    return r


In [26]:
df['age'] = df['age'].apply(lambda x: generalize_age(x, level=3))
df['workclass'] = df['workclass'].apply(lambda x: generalize_workclass(x, level=3))
df['education'] = df['education'].apply(lambda x: generalize_education(x, level=3))
df['hours-per-week'] = df['hours-per-week'].apply(lambda x: generalize_hours(x, level=3))
df['native-country'] = df['native-country'].apply(lambda x: generalize_country(x, level=3))
df['marital-status'] = df['marital-status'].apply(lambda x: generalize_marital(x, level=3))
df['occupation'] = df['occupation'].apply(lambda x: generalize_occupation(x, level=3))
df['relationship'] = df['relationship'].apply(lambda x: generalize_relationship(x, level=3))
df['race'] = df['race'].apply(lambda x: generalize_race(x, level=3))

df

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,50-59,Self-employed,83311,High,13,With partner,White-collar,With family,Majority,Male,0,0,Part-time,US,<=50K
1,30-39,Private,215646,Low,9,Without partner,Blue-collar,Without family,Majority,Male,0,0,Full-time,US,<=50K
2,50-59,Private,234721,Low,7,With partner,Blue-collar,With family,Minority,Male,0,0,Full-time,US,<=50K
3,20-29,Private,338409,High,13,With partner,White-collar,With family,Minority,Female,0,0,Full-time,Non-US,<=50K
4,30-39,Private,284582,High,14,With partner,White-collar,With family,Majority,Female,0,0,Full-time,US,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32555,20-29,Private,257302,Medium,12,With partner,White-collar,With family,Majority,Female,0,0,Full-time,US,<=50K
32556,40-49,Private,154374,Low,9,With partner,Blue-collar,With family,Majority,Male,0,0,Full-time,US,>50K
32557,50-59,Private,151910,Low,9,Without partner,White-collar,Without family,Majority,Female,0,0,Full-time,US,<=50K
32558,20-29,Private,201490,Low,9,Without partner,White-collar,With family,Majority,Male,0,0,Part-time,US,<=50K


In [30]:
def avg_equiv_class_size_metric(data, qids, k):
    """
    Return the average sizes of the equivalence classes with respect to the QID set.
    Input:
        data: The input k-anonymized dataframe
        qids: the set of QIDs
    """
    eq_classes = data.groupby(list(qids)).size()
    return eq_classes.mean() / k


def discernability_metrics(data, qids):
    """
    Return the discernability score of the dataset with respect to the QID set.
    The discernability is calculated by assigning the penalty to each tuple depending 
    on the how many tuples are indistinguishable from it.
    Input:
        data (pandas.DataFrame): The input k-anonymized dataframe
        qids (set): the set of QIDs
    """
    eq_classes = data.groupby(list(qids)).size()
    return (eq_classes ** 2).sum()


def classification_metrics(data, qids, sensitive_attr):
    """
    Return the classification metric score of the data with respect to the QID set,
    where we assign a penalty to each tuple t. If t's sensitive attribute matches 
    the majority sensitive attribute, the penalty = 0. Otherwise, penalty = size of the equivalence class.
    Input:
        data (pandas.DataFrame): The input k-anonymized dataframe
        qids (set): the set of QIDs
    """
    penalties = 0
    for _, group in data.groupby(list(qids)):
        # class_size = len(group)
        majority = group[sensitive_attr].value_counts().idxmax()
        mismatches = group[group[sensitive_attr] != majority]
        penalties += len(mismatches)
    return penalties / len(data)



QIDs = {"age", "workclass", "education", "marital-status", "occupation", "race", "sex", "native-country"}

k = 3  # k-anonymized dataset (each group size >= 3)

# --- Run metrics ---

print("Average Equivalence Class Size Metric:", avg_equiv_class_size_metric(df, QIDs, k=6))
print("Discernability Metric:", discernability_metrics(df, QIDs))
print("Classification Metric:", classification_metrics(df, QIDs, "income"))

Average Equivalence Class Size Metric: 2.6394293125810635
Discernability Metric: 5619462
Classification Metric: 0.15988943488943488
